In [107]:
%pip install datasets torch scikit-learn pandas matplotlib
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from datasets import load_dataset


ds = load_dataset("harvardairobotics/Harvard-GF", "default")

Note: you may need to restart the kernel to use updated packages.


In [108]:
#df = ds['train'].to_pandas()
#pd.set_option('display.max_rows', None)
#print(df['age'].describe())
#df['age'].plot.hist(bins=20, edgecolor='black')
#plt.show()


In [109]:
ds = load_dataset("harvardairobotics/FairVLMed", "default")

df = ds['train'].to_pandas()
X = df[['age','gender','race']].copy()
y = df['glaucoma'].copy()

X['age'] = X['age'].astype(int)
X_encoded = pd.get_dummies(X, columns=['gender', 'race'], drop_first=True)
y_encoded = df['glaucoma'].map({"yes": 1, "no": 0}).astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train.astype(float), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy().astype(float), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.astype(float), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy().astype(float), dtype=torch.float32)


print(y_train_tensor.shape)


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)




torch.Size([8000])


In [110]:
# define multi-layer perceptron

class MultiLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        # flatten input?
        return self.network(x)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiLayerPerceptron().to(device)
print(model)

MultiLayerPerceptron(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [111]:
# define training loop

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_loader, criterion, optimizer, device, epochs):
    model.train()
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


train(model, train_loader, criterion, optimizer, device, epochs=10)

Epoch [1/10], Loss: 0.6139
Epoch [2/10], Loss: 0.6441
Epoch [3/10], Loss: 0.5690
Epoch [4/10], Loss: 0.6279
Epoch [5/10], Loss: 0.6532
Epoch [6/10], Loss: 0.6649
Epoch [7/10], Loss: 0.6606
Epoch [8/10], Loss: 0.7383
Epoch [9/10], Loss: 0.7410
Epoch [10/10], Loss: 0.5495


In [112]:
def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_samples = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels.long())
            total_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)
    return total_loss / total_samples

evaluate(model, test_loader, criterion, device)

0.6229466805458069